# 06 — RAG Retriever
Load (or build) the CourseRetriever FAISS index, run sample queries, and confirm correct courses are retrieved.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("../..")
sys.path.insert(0, str(REPO_ROOT / "src"))

INDEX_DIR  = REPO_ROOT / "data" / "domain"
CATALOG    = REPO_ROOT / "data" / "course-descriptions.json"

## Build index if not present

In [ ]:
if not (INDEX_DIR / "courses.faiss").exists():
    print("Index not found — building from catalog...")
    from retrieval.ingestor import CatalogIngestor
    ingestor = CatalogIngestor(
        catalog_path=str(CATALOG),
        output_dir=str(INDEX_DIR)
    )
    ingestor.save()
    print("Index built.")
else:
    print("Index found — loading from disk.")

## Load Retriever

In [ ]:
from retrieval.retriever import CourseRetriever

retriever = CourseRetriever(index_dir=str(INDEX_DIR))
print(f"Loaded {len(retriever.metadata)} courses into retriever.")

## Sample Queries — Top-k Results with Similarity Scores

In [ ]:
queries = [
    "What courses do I need to take for software engineering?",
    "What are the prerequisites for data structures?",
    "I want to learn machine learning, what courses are available?",
    "What calculus courses does WSU offer?",
    "How do I satisfy the UCORE requirement for writing?",
]

TOP_K = 5

for query in queries:
    print(f"\nQuery: {query}")
    print("-" * 60)
    results = retriever.search(query, top_k=TOP_K)
    for i, r in enumerate(results, 1):
        print(f"  {i}. [{r['score']:.4f}] {r['course_code']} — {r['chunk_text'][:120]}...")

## Confirm Known Lookups

In [ ]:
known_checks = [
    ("data structures query",   "data structures and algorithms",  "CPTS 223"),
    ("intro programming query", "introduction to programming",     "CPTS 121"),
    ("calculus query",          "differential calculus",           "MATH 171"),
]

print("Retrieval sanity checks:")
for label, query, expected_code in known_checks:
    results = retriever.search(query, top_k=5)
    codes = [r["course_code"] for r in results]
    found = any(expected_code in c for c in codes)
    status = "PASS" if found else "MISS"
    print(f"  [{status}] {label}: expected {expected_code} in top-5 -> got {codes}")